<a href="https://colab.research.google.com/github/aiyman14/Sch-Mgmt-661-Applications-of-AI-Models/blob/main/airbnb_miniproject_AA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Airbnb Mini-Project: Price Prediction with Neural Networks
**SCH-MGMT 661: Applications of AI Models**

This project extends Assignment 1 by comparing different neural network architectures and generating business-relevant insights for Airbnb price prediction.

---
## 1. Data Preparation
Reusing the cleaned dataset from Assignment 1 with additional features.

### 1.1 Data Loading and Initial Exploration

In [ ]:
# Import required libraries for data analysis and visualization

import pandas as pd              # for data manipulation
import numpy as np               # for numerical operations
import matplotlib.pyplot as plt  # for plotting
import seaborn as sns            # for enhanced visualizations

# Set a default aesthetic style for plots
sns.set(style="whitegrid")

In [ ]:
# import Airbnb Listing datasets for Asheville

listings_url = 'https://data.insideairbnb.com/united-states/nc/asheville/2024-06-21/data/listings.csv.gz'

# Load the datasets into DataFrames
listings_df = pd.read_csv(listings_url, compression='gzip')

Loading the dataset for data exploration and cleaning. Taken directly from Assignment 1.

In [ ]:
# Display the first 10 rows
listings_df.head(10)

In [ ]:
# Explore columns, data types, and non-null counts
listings_df.info()

In [ ]:
# Display the shape of the DataFrame (rows, columns)
listings_df.shape

Initial data exploration. Taken directly from Assignment 1.

### 1.2 Filter Listings for Asheville

In [ ]:
# Print all unique host locations in the dataset
print(listings_df['host_location'].unique())      # List all location names
print(len(listings_df['host_location'].unique())) # Total number of unique locations

In [ ]:
# Filter Listings for Asheville
asheville_df = listings_df[listings_df['host_location'] == 'Asheville, NC']

In [ ]:
# Shape of the filtered dataset
asheville_df.shape

Ensure listings belong to hosts located in Asheville, NC. Kept all records with host_location as Asheville, dropped all rows with missing or different host_location values. Taken directly from Assignment 1.

### 1.3 Select Relevant Features

In [ ]:
# Define the columns we want to keep and create a new dataframe with the selected columns
selected_columns = [
    'price', 'bathrooms','bedrooms', 'number_of_reviews',
    'room_type', 'host_identity_verified', 'host_is_superhost',
    'accommodates', 'review_scores_rating'
]

asheville_df = asheville_df[selected_columns]

In [ ]:
# Preview the first few rows of the new dataframe
asheville_df.head()

Extended the selected features from W3 to include accommodates, review_scores_rating, and room_type. Taken directly from Assignment 1.

### 1.4 Handle Missing Values

In [ ]:
# Check for missing values in each column
asheville_df.isnull().sum()

Check for missing values before applying imputation strategies. Taken directly from Assignment 1.

In [ ]:
# Drop rows where the target variable (price) is missing and then re-check for missing values
asheville_df = asheville_df.dropna(subset=['price'])
asheville_df.isnull().sum()

Drop rows where the price value is missing. Taken directly from Assignment 1.

In [ ]:
# Impute host_is_superhost using mode
asheville_df['host_is_superhost'] = asheville_df['host_is_superhost'].fillna(
    asheville_df['host_is_superhost'].mode()[0]
)
asheville_df.isnull().sum()

Impute host_is_superhost using mode. Taken directly from Assignment 1.

In [ ]:
# Impute numerical missing values using mean or median, depending on data distribution
asheville_df['bedrooms'] = asheville_df['bedrooms'].fillna(asheville_df['bedrooms'].median())
asheville_df['bathrooms'] = asheville_df['bathrooms'].fillna(asheville_df['bathrooms'].mean())
asheville_df.isnull().sum()

Impute bedrooms using median and bathrooms using mean to fill missing values for numerical features except for review_scores_rating. Taken directly from Assignment 1.

In [ ]:
# For review_scores_rating, replace missing values with 0 (indicating no reviews)
asheville_df['review_scores_rating'] = asheville_df['review_scores_rating'].fillna(0)

# Create a dummy variable (has_review_scores) to indicate whether a listing has a review
asheville_df['has_review_scores'] = (asheville_df['review_scores_rating'] > 0).astype(int)

asheville_df.isnull().sum()

Replace missing values of review_scores_rating with 0. Then created a dummy variable has_review_scores to indicate whether a listing has a review. has_review_scores = 1 if review_scores_rating > 0, has_review_scores = 0 if review_scores_rating == 0 (no reviews). Taken directly from Assignment 1.

### 1.5 Fix Data Types and Encode Columns

In [ ]:
# Check data types
asheville_df.dtypes

In [ ]:
# Convert price to Numeric
asheville_df['price'] = asheville_df['price'].replace(r'[\$,]', '', regex=True).astype(float)

Convert price from string to numeric for analysis. Taken directly from Assignment 1.

In [ ]:
# Convert Boolean-like Columns
asheville_df['host_identity_verified'] = asheville_df['host_identity_verified'].map({'t': 1, 'f': 0})
asheville_df['host_is_superhost'] = asheville_df['host_is_superhost'].map({'t': 1, 'f': 0})

Dummy encode categorical variables host_identity_verified and host_is_superhost. Taken directly from Assignment 1.

In [ ]:
# Encode Categorical Column

room_type_dummies = pd.get_dummies(asheville_df['room_type'], prefix='room_type', drop_first=True)
room_type_dummies = room_type_dummies.astype(int)

# Add new columns to the DataFrame
asheville_df = pd.concat([asheville_df, room_type_dummies], axis=1)
asheville_df.drop('room_type', axis=1, inplace=True)

asheville_df.head()

One-hot encode room_type due to multi-category. Taken directly from Assignment 1.

### 1.6 Handle Outliers

In [ ]:
# Visualize the original price distribution
plt.figure(figsize=(10, 4))
sns.histplot(asheville_df['price'], bins=50, kde=True, color='green')
plt.title("Price Distribution (Before Removing Top 1%)")
plt.xlabel("Price")
plt.show()

In [ ]:
# Remove top 1% of extreme price values
price_cap = asheville_df['price'].quantile(0.99)
asheville_df = asheville_df[asheville_df['price'] <= price_cap]

In [ ]:
# Visualize the cleaned price distribution
plt.figure(figsize=(10, 4))
sns.histplot(asheville_df['price'], bins=50, kde=True, color='green')
plt.title("Price Distribution (After Removing Top 1%)")
plt.xlabel("Price")
plt.show()

Handle outliers by removing the top 1% of extreme price values. Taken directly from Assignment 1.

### 1.7 Descriptive Statistics

In [ ]:
# Descriptive Statistics
asheville_df.describe()

---
## 2. Additional Features
New features added for the mini-project to enhance model performance. These are pulled from the original dataset (`listings_df`) using index alignment with our cleaned `asheville_df`.

In [ ]:
# Reference the full Asheville data from the original dataset (still in memory)
# Using index alignment so only rows that survived cleaning get new feature values
asheville_full = listings_df[listings_df['host_location'] == 'Asheville, NC']

In [ ]:
# Feature 1: amenities_count — count the number of amenities listed for each property
# The amenities column is stored as a string like '["Wifi", "Kitchen", "Free parking"]'
# We parse it by splitting on commas to get the count
asheville_df['amenities_count'] = asheville_full['amenities'].apply(
    lambda x: len(str(x).split(',')) if pd.notna(x) else 0
)

# Feature 2: host_total_listings_count — total number of properties managed by the host
asheville_df['host_total_listings_count'] = asheville_full['host_total_listings_count']

# Feature 3: minimum_nights — minimum stay requirement for the listing
asheville_df['minimum_nights'] = asheville_full['minimum_nights']

In [ ]:
# Check for any missing values in the new features
print(asheville_df[['amenities_count', 'host_total_listings_count', 'minimum_nights']].isnull().sum())
print()

# Impute any missing values with median (safe default for skewed numeric data)
for col in ['amenities_count', 'host_total_listings_count', 'minimum_nights']:
    asheville_df[col] = asheville_df[col].fillna(asheville_df[col].median())

In [ ]:
# Quick summary of the new features
asheville_df[['amenities_count', 'host_total_listings_count', 'minimum_nights']].describe()

In [ ]:
# Preview the updated dataframe with new features
asheville_df.head()

### Why These Features Were Chosen

Three additional features were selected to enhance the model beyond the Assignment 1 baseline:

**amenities_count** — Captures the overall "value offering" of a property. More amenities generally means a higher-quality experience and higher nightly rates.

**host_total_listings_count** — Distinguishes professional hosts (many properties) from individual hosts. Professional hosts often have different pricing strategies.

**minimum_nights** — Reflects pricing strategy. Short minimum stays (1–2 nights) target tourists at higher per-night rates, while longer minimums often offer discounted rates.

All three are fully populated in the raw dataset, avoiding additional imputation noise. Together they add property quality, host behavior, and booking strategy dimensions not captured by the Assignment 1 features.

### Brief EDA on New Features

In [ ]:
# Histograms of the new features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(asheville_df['amenities_count'], bins=30, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Amenities Count Distribution')

sns.histplot(asheville_df['host_total_listings_count'], bins=30, kde=True, color='coral', ax=axes[1])
axes[1].set_title('Host Total Listings Count Distribution')

sns.histplot(asheville_df['minimum_nights'], bins=30, kde=True, color='seagreen', ax=axes[2])
axes[2].set_title('Minimum Nights Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix including new features
corr = asheville_df.corr()
plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix (Including New Features)")
plt.show()

Brief overview of the new features. Amenities count and host_total_listings_count are right-skewed, indicating most listings have a moderate number of amenities and most hosts manage few properties. Minimum nights is heavily concentrated at low values (1–3 nights). The correlation matrix shows how the new features relate to price and to each other compared to the Assignment 1 features.

---
## 3. Modeling Preparation
Feature selection, train/test split, normalization, and helper functions.

In [ ]:
# Imports required libraries for pre-processing and modeling

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Normalization, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Baseline features (same as Assignment 1 — used for Model A and Model B)
features = ['bedrooms', 'bathrooms', 'number_of_reviews', 'host_is_superhost',
            'host_identity_verified', 'accommodates', 'review_scores_rating',
            'has_review_scores', 'room_type_Hotel room', 'room_type_Private room',
            'room_type_Shared room']
target = 'price'

# Enhanced features for Model C (baseline + new features)
features_enhanced = features + ['amenities_count', 'host_total_listings_count', 'minimum_nights']

Selecting baseline features (same as Assignment 1) for Models A and B, and an enhanced feature set including the new features for Model C.

In [ ]:
# Define X and y for baseline models
X = asheville_df[features]
y = asheville_df[target]

# Train-Test Split (80/20, same as Assignment 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define and Adapt Normalization Layer for baseline features
norm_layer = Normalization()
norm_layer.adapt(X_train.values)

In [ ]:
# Define X and y for enhanced model (Model C)
# Use SAME train/test rows as baseline for fair comparison
X_enhanced = asheville_df[features_enhanced]

X_train_enh = X_enhanced.loc[X_train.index]
X_test_enh = X_enhanced.loc[X_test.index]

# Define and Adapt Normalization Layer for enhanced features
norm_layer_enh = Normalization()
norm_layer_enh.adapt(X_train_enh.values)

Train-test split with 80/20 ratio and random_state=42, same as Assignment 1. Separate normalization layers are adapted for baseline and enhanced feature sets. Enhanced model uses the same train/test row indices to ensure fair comparison.

In [ ]:
# Helper function to evaluate a model on the test set
def eval_on_test(m, X_te, y_te):
    y_pred = m.predict(X_te, verbose=0).flatten()
    mae  = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    r2   = r2_score(y_te, y_pred)
    return mae, rmse, r2

In [ ]:
# Helper function to plot training vs validation curves
def plot_training_curves(history, model_name="Model"):
    """Plots training vs validation MAE and MSE curves for a Keras model."""
    # Plot MAE
    plt.figure(figsize=(6, 4))
    plt.plot(history.history['mae'], label='Train MAE')
    plt.plot(history.history['val_mae'], label='Validation MAE')
    plt.xlabel('Epochs')
    plt.ylabel('MAE (Price)')
    plt.title(f'{model_name}: Training vs Validation MAE')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Plot Loss (MSE)
    plt.figure(figsize=(6, 4))
    plt.plot(history.history['loss'], label='Train Loss (MSE)')
    plt.plot(history.history['val_loss'], label='Validation Loss (MSE)')
    plt.xlabel('Epochs')
    plt.ylabel('MSE')
    plt.title(f'{model_name}: Training vs Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

Helper functions for model evaluation and plotting, following the miniproject notebook structure. `eval_on_test` returns MAE, RMSE, and R² on the test set. `plot_training_curves` generates training vs validation MAE and loss (MSE) plots.

---
## 4. Model A: Baseline Neural Network (Assignment 1 Model)
Input → Dense(64, ReLU) → Dense(1)

In [ ]:
# Build Model A (Baseline — same architecture as Assignment 1)
model_a = Sequential([
    Input(shape=(X_train.shape[1],)),
    norm_layer,
    Dense(64, activation='relu'),
    Dense(1)  # Regression output
])

# Compile Model A
model_a.compile(optimizer='adam', loss='mse', metrics=['mae'])

model_a.summary()

In [ ]:
# Train Model A
hist_a = model_a.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=70,
    verbose=1
)

In [ ]:
# Evaluate Model A on test set
mae_a, rmse_a, r2_a = eval_on_test(model_a, X_test, y_test)
print(f"Model A (Baseline) -> MAE: {round(mae_a, 2)}  RMSE: {round(rmse_a, 2)}  R²: {round(r2_a, 2)}")

In [ ]:
# Plot training curves for Model A
plot_training_curves(hist_a, model_name="Model A (Baseline)")

Model A is the baseline from Assignment 1. Single hidden layer with 64 nodes and ReLU activation, trained for 70 epochs with 20% validation split.

---
## 5. Model B: Deeper Neural Network
*Input → Dense(128, ReLU) → Dense(64, ReLU) → Dense(1)*

*(To be added)*

---
## 6. Model C: Enhanced Features + Early Stopping
*Includes new features + Input → Dense(128, ReLU) → Dense(64, ReLU) → Dense(1) + EarlyStopping*

*(To be added)*

---
## 7. Model Comparison
*Side-by-side metrics (MAE, MSE, R²) and learning curves for all three models.*

*(To be added)*

---
## 8. Discussion and Business Insights
*Which model performs best? Did new features improve predictions? Other insights.*

*(To be added)*